# MODIS Burned Area (MBA) - Yearly Metrics

**Autor:** Ruben Butzer  
**Version:** 2.0  
**Letzte Aktualisierung:** 18. November 2025

---

## Übersicht
Dieses Notebook erstellt jährliche Brandmetriken aus MODIS Burned Area Collection (MCD64A1) für den Zeitraum 2000-2025.

## Datenquelle
- **Dataset**: MODIS/061/MCD64A1
- **Zeitraum**: 2000-2025
- **Auflösung**: 500m
- **CRS**: EPSG:3035 (LAEA Europe)

## Generierte Metriken (pro Jahr)
1. **burned**: Binär-Maske (0/1) - Pixel brannte im Jahr
2. **count**: Anzahl der Brandtage (DOY-Werte)
3. **doy_1 bis doy_4**: doy von max. 4 Brand-DOYs im Pixel (0 wenn weniger als 4 Brände, maximum nie höher als 4)
4. **doy_min**: Erster Brandtag (DOY)
5. **doy_max**: Letzter Brandtag (DOY)
6. **doy_span**: Zeitspanne zwischen erstem und letztem Brand
7. **month_first**: Monat des ersten Brandes
8. **month_last**: Monat des letzten Brandes
9. **season_length**: Länge der Brandsaison in Monaten

## Workflow
1. Daten aus Google Earth Engine exportieren
2. Von Google Drive herunterladen
3. Mosaic aus Kacheln erstellen
4. Bandbeschreibungen hinzufügen

## Output
- **Format**: Multi-band GeoTIFF (312 Bänder = 26 Jahre × 12 Metriken)
- **Ordner**: `_Runs/01_Fire_Metrics/MBA_500m_3035_DOY_season`

---

In [31]:
import ee
import geemap
import ipywidgets as widgets
from IPython.display import display
import os
from osgeo import gdal, ogr
import rasterio
import geopandas as gpd
import numpy as np
import time
from tqdm import tqdm
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
import getpass


# Earth Engine initialisieren
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()


Successfully saved authorization token.


In [59]:
#workDir = r"\\141.20.141.12\SAN_BioGeo\_HiWi\Ruben\thesis"
workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"

user = getpass.getuser()
if user == "butzerfe":
    drive_folder        = "GEE_downloads"
    drive_folder_url    = "1hiXt6bvYpxDTUaverk76f_D9ZkJzm_Ld"
    GoogleAuth.DEFAULT_SETTINGS['client_config_file'] = "L:/_HiWi/Ruben/wildE/client_secrets_ruben_geopy.json"

 # PyDrive Authentifizierung
    gauth = GoogleAuth()
    gauth.LocalWebserverAuth()
    drive = GoogleDrive(gauth)
    print("Google Drive authentication successful.")

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?client_id=579762042005-bktd78digk65s9sht4hho28rq5s30apo.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&access_type=offline&response_type=code

Authentication successful.
Google Drive authentication successful.


In [3]:
#load in grid
grid_ee = ee.FeatureCollection("projects/ee-butzerfe/assets/wilde_only_grid_glance_eu_150km")
#grid_ee = grid_ee.filter(ee.Filter.eq('wildE', 1))

In [4]:

# check crs
grid_ee_crs = grid_ee.first().geometry().projection().getInfo()
grid_ee_crs


{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}

In [ ]:

# transform to 3035 LAEA
def to_laea(feat):
    return feat.transform('EPSG:3035', 1)

grid_ee = grid_ee.map(to_laea)

# map = geemap.Map(center=[50, 10], zoom=4)
# map.addLayer(grid_ee, {}, "Grid")
# map.addLayerControl()
# map 

In [6]:
grid_ee_crs = grid_ee.first().geometry().projection().getInfo()
print("Grid CRS:", grid_ee_crs)

Grid CRS: {'type': 'Projection', 'crs': 'EPSG:3035', 'transform': [1, 0, 0, 0, 1, 0]}


In [63]:
# Test: Kleiner Export für ein einzelnes Jahr
test_year = 2020
test_region = grid_ee.geometry()
test_scale = 5000  # Größere Pixel für schnelleren Test

# Erstelle ein kleines Test-Image (nur ein Jahr)
yearly = firemodis.filterDate(f'{test_year}-01-01', f'{test_year}-12-31').select('BurnDate')
masked = yearly.map(lambda img: img.updateMask(img.gt(0)))
count = masked.count().rename(f"{test_year}_count").toUint16()
burned = count.gt(0).rename(f"{test_year}_burned").toUint16()

# Grid-Maske
mask = ee.Image(0).byte().paint(grid_ee, 1)
test_img = ee.Image.cat([burned, count]).toUint16().updateMask(mask)

# Test-Export
test_folder = os.path.join(workDir, "_Runs/01_Fire_Metrics/TEST_GDrive")
if not os.path.exists(test_folder):
    os.makedirs(test_folder)

# Verwende einen EINDEUTIGEN Ordnernamen mit Timestamp
import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
export_folder_name = f"GEE_MBA_{timestamp}"

task = ee.batch.Export.image.toDrive(
    image=test_img,
    description='TEST_MBA_GDrive',
    folder=export_folder_name,  # EINDEUTIGER Name
    fileNamePrefix='TEST_MBA_GDrive',
    region=test_region,
    crs='EPSG:3035',
    scale=test_scale,
    maxPixels=1e10,
    fileFormat='GeoTIFF'
)
task.start()
print(f"Test-Export gestartet: {task.id}")
print(f"Export-Ordner Name: {export_folder_name}")

# Warten auf Completion (mit Timeout)
import time
max_wait = 300  # 5 Minuten
start_time = time.time()

while task.active():
    elapsed = time.time() - start_time
    print(f"Export läuft... ({int(elapsed)}s)")
    if elapsed > max_wait:
        print("Timeout erreicht!")
        break
    time.sleep(10)

status = task.status()
print(f"\nFinal Status: {status['state']}")

if status['state'] == 'COMPLETED':
    print("✓ Export erfolgreich!")
    print("\nStarte Download von Google Drive...")
    
    # Warte kurz, damit die Datei auf GDrive verfügbar ist
    time.sleep(5)
    
    # Suche nach dem von GEE erstellten Ordner
    folder_query = f"title='{export_folder_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
    folder_list = drive.ListFile({'q': folder_query}).GetList()
    
    if len(folder_list) == 0:
        print(f"✗ Ordner '{export_folder_name}' nicht gefunden!")
        print("Suche nach allen GEE-Ordnern...")
        all_folders = drive.ListFile({'q': "mimeType='application/vnd.google-apps.folder' and trashed=false"}).GetList()
        for f in all_folders[:10]:
            print(f"  - {f['title']} (ID: {f['id']})")
    else:
        gee_folder_id = folder_list[0]['id']
        print(f"✓ Ordner gefunden: {export_folder_name} (ID: {gee_folder_id})")
        
        # Download von GDrive
        drive_list = drive.ListFile({'q': f"'{gee_folder_id}' in parents and trashed=false"}).GetList()
        
        print(f"Anzahl Dateien im Ordner: {len(drive_list)}")
        
        downloaded = False
        for file in drive_list:
            if 'TEST_MBA_GDrive' in file['title']:
                file_id = drive.CreateFile({'id': file['id']})
                fname = file["title"]
                outName = os.path.join(test_folder, fname)
                
                print(f"Lade herunter: {fname}")
                file_id.GetContentFile(outName)
                print(f"✓ Gespeichert: {outName}")
                
                # Datei aus GDrive löschen
                file_id.Delete()
                print(f"✓ Datei aus Google Drive gelöscht")
                downloaded = True
        
        # Lösche den leeren Ordner
        if downloaded:
            folder_obj = drive.CreateFile({'id': gee_folder_id})
            folder_obj.Delete()
            print(f"✓ Ordner '{export_folder_name}' gelöscht")
            
            print("\n✓✓✓ TEST ERFOLGREICH! ✓✓✓")
            print(f"Datei liegt in: {test_folder}")
        else:
            print("\n✗ Keine Datei gefunden!")
        
elif status['state'] == 'FAILED':
    print(f"✗ Export fehlgeschlagen: {status.get('error_message', 'Unbekannter Fehler')}")
else:
    print(f"✗ Unerwarteter Status: {status['state']}")

Test-Export gestartet: 62VRUG3Q77XNXTYUCQ32BRBJ
Export-Ordner Name: GEE_MBA_20251118_120549
Export läuft... (0s)
Export läuft... (10s)
Export läuft... (20s)
Export läuft... (30s)
Export läuft... (41s)
Export läuft... (51s)
Export läuft... (61s)
Export läuft... (71s)

Final Status: COMPLETED
✓ Export erfolgreich!

Starte Download von Google Drive...
✓ Ordner gefunden: GEE_MBA_20251118_120549 (ID: 1IpTxqv7extwR1HCCRnT9YHoEvAEcSjg6)
Anzahl Dateien im Ordner: 1
Lade herunter: TEST_MBA_GDrive.tif
✓ Gespeichert: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs/01_Fire_Metrics/TEST_GDrive\TEST_MBA_GDrive.tif
✓ Datei aus Google Drive gelöscht
✓ Ordner 'GEE_MBA_20251118_120549' gelöscht

✓✓✓ TEST ERFOLGREICH! ✓✓✓
Datei liegt in: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs/01_Fire_Metrics/TEST_GDrive


### (alte Version) Get Yearly MBA of Burned Area from MODIS MCD64A1 including DOY of Burned Area

In [ ]:
# # MODIS Burned Area Collection
# firemodis = ee.ImageCollection("MODIS/061/MCD64A1")
# start_year = 2000
# end_year = 2025
# years = list(range(start_year, end_year + 1))

# # Region und Grid definieren
# region = grid_ee.geometry()
# scale = 500

# import datetime

# def burned_metrics_year(year):
#     yearly = firemodis.filterDate(f'{year}-01-01', f'{year}-12-31').select('BurnDate')
#     # Anzahl Brand-Frames pro Pixel (als UInt16)
#     count = yearly.map(lambda img: img.gt(0).toUint16()) \
#                   .sum() \
#                   .rename(f"{year}_count").toUint16()

#     # Max und Min DOY (0 = kein Brand) als UInt16
#     masked = yearly.map(lambda img: img.updateMask(img.gt(0)))
#     doy_max = masked.reduce(ee.Reducer.max()).rename(f"{year}_doy_max").unmask(0).toUint16()
#     doy_min = masked.reduce(ee.Reducer.min()).rename(f"{year}_doy_min").unmask(0).toUint16()

#     # Binär-Band als UInt16 (0/1)
#     burned = count.gt(0).rename(f"{year}_burned").toUint16()

#     # Monat des ersten/letzten Feuers (0 = kein Brand) -> ebenfalls UInt16
#     doy_list = list(range(1, 367))
#     month_list = [(datetime.date(year, 1, 1) + datetime.timedelta(days=d - 1)).month for d in doy_list]
#     month_first = doy_min.remap(doy_list, month_list, 0).rename(f"{year}_month_first").toUint16()
#     month_last  = doy_max.remap(doy_list, month_list, 0).rename(f"{year}_month_last").toUint16()

#     # Auf das Grid maskieren (wie vorher)
#     mask = ee.Image(0).byte().paint(grid_ee, 1)
#     img = ee.Image.cat([burned, count, doy_min, doy_max, month_first, month_last]).toUint16()
#     return img.updateMask(mask)

# # Liste von Jahresbildern (je Jahr mehrere Bänder)
# burned_images = [burned_metrics_year(year) for year in years]
# burned_stack = ee.Image.cat(burned_images)

# # Export als Multiband-GeoTIFF
# DL_folder = os.path.join(workDir, "_Runs/01_Fire_Metrics/MBA_500m_3035_DOY")
# if not os.path.exists(DL_folder):
#     os.makedirs(DL_folder)

# task = ee.batch.Export.image.toDrive(
#     image=burned_stack,
#     description='MBA_burned_metrics_yearly',
#     folder='GEE',
#     fileNamePrefix='MBA_burned_metrics_yearly',
#     region=region,
#     crs='EPSG:3035',
#     scale=scale,
#     maxPixels=1e13,
#     fileFormat='GeoTIFF'
# )
# task.start()
# print("Export gestartet. Überprüfe den Task-Status in der Earth Engine-Taskliste.")
# # ...existing code...

# # Warten bis Export abgeschlossen ist
# while task.active():
#     print("Export läuft...")
#     time.sleep(30)

# if task.status()['state'] == 'COMPLETED':
#     print("Export erfolgreich abgeschlossen! Starte Download vom Google Drive...")
# else:
#     print("Export fehlgeschlagen:", task.status())

# # Download processed tiles from GDrive
# drive_list = drive.ListFile({'q': f"'{drive_folder_url}' in parents and trashed=false"}).GetList()
# for file in tqdm(drive_list):
#     file_id = drive.CreateFile({'id': file['id']})
#     fname = file["title"]
#     outName = os.path.join(DL_folder, fname)
#     file_id.GetContentFile(outName)
#     file_id.Delete()
#     print(f"{fname} wurde nach {outName} heruntergeladen und aus Google Drive gelöscht.")

# print("Alle verfügbaren Dateien wurden heruntergeladen.")


Export gestartet. Überprüfe den Task-Status in der Earth Engine-Taskliste.
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export erfolgreich abgeschlossen! Starte Download vom Google Drive...


0it [00:00, ?it/s]

Alle verfügbaren Dateien wurden heruntergeladen.


### (aktuelle Version) updated MBA metrics with exact number of DOYs per year, doy_span and season_length

In [ ]:
# MODIS Burned Area Collection
firemodis = ee.ImageCollection("MODIS/061/MCD64A1")
start_year = 2000
end_year = 2025
years = list(range(start_year, end_year + 1))

# Region und Grid definieren
region = grid_ee.geometry()
scale = 500

import datetime

def burned_metrics_year(year):
    yearly = firemodis.filterDate(f'{year}-01-01', f'{year}-12-31').select('BurnDate')
    
    # Maskiere nur Pixel MIT Brand (BurnDate > 0)
    masked = yearly.map(lambda img: img.updateMask(img.gt(0)))
    
    # Erstelle Array mit allen DOY-Werten pro Pixel
    doy_array = masked.toArray()  # Shape: [time]
    
    # Extrahiere erste 4 Werte aus dem Array mit arraySlice
    doy_1 = doy_array.arraySlice(0, 0, 1).arrayProject([0]).arrayFlatten([['doy_1']]).unmask(0).toUint16().rename(f"{year}_doy_1")
    doy_2 = doy_array.arraySlice(0, 1, 2).arrayProject([0]).arrayFlatten([['doy_2']]).unmask(0).toUint16().rename(f"{year}_doy_2")
    doy_3 = doy_array.arraySlice(0, 2, 3).arrayProject([0]).arrayFlatten([['doy_3']]).unmask(0).toUint16().rename(f"{year}_doy_3")
    doy_4 = doy_array.arraySlice(0, 3, 4).arrayProject([0]).arrayFlatten([['doy_4']]).unmask(0).toUint16().rename(f"{year}_doy_4")
    
    # Count (Anzahl nicht-null Werte)
    count = masked.count().rename(f"{year}_count").toUint16()
    
    # Min und Max DOY
    doy_min = masked.reduce(ee.Reducer.min()).unmask(0).toUint16().rename(f"{year}_doy_min")
    doy_max = masked.reduce(ee.Reducer.max()).unmask(0).toUint16().rename(f"{year}_doy_max")
    
    # DOY Span
    doy_span = doy_max.subtract(doy_min).rename(f"{year}_doy_span").toUint16()
    
    # Binär-Band
    burned = count.gt(0).rename(f"{year}_burned").toUint16()
    
    # Monate
    doy_list = list(range(1, 367))
    month_list = [(datetime.date(year, 1, 1) + datetime.timedelta(days=d - 1)).month for d in doy_list]
    month_first = doy_min.remap(doy_list, month_list, 0).rename(f"{year}_month_first").toUint16()
    month_last = doy_max.remap(doy_list, month_list, 0).rename(f"{year}_month_last").toUint16()
    
    # Season Length
    season_length = ee.Image(0).where(
        count.eq(0), 0
    ).where(
        count.gt(0).And(month_first.eq(month_last)), 1
    ).where(
        count.gt(0).And(month_first.neq(month_last)), 
        month_last.subtract(month_first).add(1)
    ).rename(f"{year}_season_length").toUint16()
    
    # Grid-Maske
    mask = ee.Image(0).byte().paint(grid_ee, 1)
    img = ee.Image.cat([
        burned, 
        count, 
        doy_1, 
        doy_2, 
        doy_3, 
        doy_4,
        doy_min,
        doy_max,
        doy_span,
        month_first, 
        month_last,
        season_length
    ]).toUint16()
    
    return img.updateMask(mask)

# Liste von Jahresbildern (je Jahr 10 Bänder)
burned_images = [burned_metrics_year(year) for year in years]
burned_stack = ee.Image.cat(burned_images)

# Export als Multiband-GeoTIFF
DL_folder = os.path.join(workDir, "_Runs/01_Fire_Metrics/MBA_500m_3035_DOY_season")
if not os.path.exists(DL_folder):
    os.makedirs(DL_folder)

task = ee.batch.Export.image.toDrive(
    image=burned_stack,
    description='MBA_burned_metrics_yearly_season_500m',
    folder='GEE',
    fileNamePrefix='MBA_burned_metrics_yearly_season_500m',
    region=region,
    crs='EPSG:3035',
    scale=scale,
    maxPixels=1e13,
    fileFormat='GeoTIFF'
)
task.start()
print("Export gestartet. Überprüfe den Task-Status in der Earth Engine-Taskliste.")

# Warten bis Export abgeschlossen ist
while task.active():
    print("Export läuft...")
    time.sleep(30)

if task.status()['state'] == 'COMPLETED':
    print("Export erfolgreich abgeschlossen! Starte Download vom Google Drive...")
else:
    print("Export fehlgeschlagen:", task.status())

# Download processed tiles from GDrive
drive_list = drive.ListFile({'q': f"'{drive_folder_url}' in parents and trashed=false"}).GetList()
for file in tqdm(drive_list):
    file_id = drive.CreateFile({'id': file['id']})
    fname = file["title"]
    outName = os.path.join(DL_folder, fname)
    file_id.GetContentFile(outName)
    file_id.Delete()
    print(f"{fname} wurde nach {outName} heruntergeladen und aus Google Drive gelöscht.")

print("Alle verfügbaren Dateien wurden heruntergeladen.")

Export gestartet. Überprüfe den Task-Status in der Earth Engine-Taskliste.
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft.

0it [00:00, ?it/s]

Alle verfügbaren Dateien wurden heruntergeladen.


In [65]:
# MODIS Burned Area Collection
firemodis = ee.ImageCollection("MODIS/061/MCD64A1")
start_year = 2000
end_year = 2025
years = list(range(start_year, end_year + 1))

# Region und Grid definieren
region = grid_ee.geometry()
scale = 50000

import datetime

def burned_metrics_year(year):
    yearly = firemodis.filterDate(f'{year}-01-01', f'{year}-12-31').select('BurnDate')
    
    # Maskiere nur Pixel MIT Brand (BurnDate > 0)
    masked = yearly.map(lambda img: img.updateMask(img.gt(0)))
    
    # Erstelle Array mit allen DOY-Werten pro Pixel
    doy_array = masked.toArray()  # Shape: [time]
    
    # Extrahiere erste 4 Werte aus dem Array mit arraySlice
    doy_1 = doy_array.arraySlice(0, 0, 1).arrayProject([0]).arrayFlatten([['doy_1']]).unmask(0).toUint16().rename(f"{year}_doy_1")
    doy_2 = doy_array.arraySlice(0, 1, 2).arrayProject([0]).arrayFlatten([['doy_2']]).unmask(0).toUint16().rename(f"{year}_doy_2")
    doy_3 = doy_array.arraySlice(0, 2, 3).arrayProject([0]).arrayFlatten([['doy_3']]).unmask(0).toUint16().rename(f"{year}_doy_3")
    doy_4 = doy_array.arraySlice(0, 3, 4).arrayProject([0]).arrayFlatten([['doy_4']]).unmask(0).toUint16().rename(f"{year}_doy_4")
    
    # Count (Anzahl nicht-null Werte)
    count = masked.count().rename(f"{year}_count").toUint16()
    
    # Min und Max DOY
    doy_min = masked.reduce(ee.Reducer.min()).unmask(0).toUint16().rename(f"{year}_doy_min")
    doy_max = masked.reduce(ee.Reducer.max()).unmask(0).toUint16().rename(f"{year}_doy_max")
    
    # DOY Span
    doy_span = doy_max.subtract(doy_min).rename(f"{year}_doy_span").toUint16()
    
    # Binär-Band
    burned = count.gt(0).rename(f"{year}_burned").toUint16()
    
    # Monate
    doy_list = list(range(1, 367))
    month_list = [(datetime.date(year, 1, 1) + datetime.timedelta(days=d - 1)).month for d in doy_list]
    month_first = doy_min.remap(doy_list, month_list, 0).rename(f"{year}_month_first").toUint16()
    month_last = doy_max.remap(doy_list, month_list, 0).rename(f"{year}_month_last").toUint16()
    
    # Season Length
    season_length = ee.Image(0).where(
        count.eq(0), 0
    ).where(
        count.gt(0).And(month_first.eq(month_last)), 1
    ).where(
        count.gt(0).And(month_first.neq(month_last)), 
        month_last.subtract(month_first).add(1)
    ).rename(f"{year}_season_length").toUint16()
    
    # Grid-Maske
    mask = ee.Image(0).byte().paint(grid_ee, 1)
    img = ee.Image.cat([
        burned, 
        count, 
        doy_1, 
        doy_2, 
        doy_3, 
        doy_4,
        doy_min,
        doy_max,
        doy_span,
        month_first, 
        month_last,
        season_length
    ]).toUint16()
    
    return img.updateMask(mask)

# Liste von Jahresbildern (je Jahr 12 Bänder)
burned_images = [burned_metrics_year(year) for year in years]
burned_stack = ee.Image.cat(burned_images)

# Export als Multiband-GeoTIFF
DL_folder = os.path.join(workDir, "_Runs/01_Fire_Metrics/MBA_50000m_3035_DOY_season")
if not os.path.exists(DL_folder):
    os.makedirs(DL_folder)

# EINDEUTIGER Ordnername mit Timestamp
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
export_folder_name = f"MBA_GEE_{timestamp}"

task = ee.batch.Export.image.toDrive(
    image=burned_stack,
    description='MBA_burned_metrics_yearly_season_50000m',
    folder=export_folder_name,  # GEÄNDERT: Eindeutiger Ordnername
    fileNamePrefix='MBA_burned_metrics_yearly_season_50000m',
    region=region,
    crs='EPSG:3035',
    scale=scale,
    maxPixels=1e13,
    fileFormat='GeoTIFF'
)
task.start()
print(f"Export gestartet: {task.id}")
print(f"Export-Ordner auf GDrive: {export_folder_name}")
print("Überprüfe den Task-Status in der Earth Engine-Taskliste.")

# Warten bis Export abgeschlossen ist
while task.active():
    print("Export läuft...")
    time.sleep(30)

status = task.status()
if status['state'] == 'COMPLETED':
    print("✓ Export erfolgreich abgeschlossen!")
    print("\nStarte automatischen Download von Google Drive...")
    
    # Kurze Wartezeit für GDrive-Synchronisation
    time.sleep(10)
    
    # Suche nach dem von GEE erstellten Ordner
    folder_query = f"title='{export_folder_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
    folder_list = drive.ListFile({'q': folder_query}).GetList()
    
    if len(folder_list) == 0:
        print(f"✗ Ordner '{export_folder_name}' nicht gefunden!")
        print("HINWEIS: Bitte manuell von Google Drive herunterladen.")
    else:
        gee_folder_id = folder_list[0]['id']
        print(f"✓ Ordner gefunden: {export_folder_name} (ID: {gee_folder_id})")
        
        # Download aller Dateien aus dem Ordner
        drive_list = drive.ListFile({'q': f"'{gee_folder_id}' in parents and trashed=false"}).GetList()
        print(f"Anzahl Kacheln: {len(drive_list)}")
        
        downloaded_count = 0
        for file in tqdm(drive_list, desc="Download"):
            file_id = drive.CreateFile({'id': file['id']})
            fname = file["title"]
            outName = os.path.join(DL_folder, fname)
            
            # Download
            file_id.GetContentFile(outName)
            
            # Aus GDrive löschen
            file_id.Delete()
            downloaded_count += 1
        
        # Lösche den leeren Ordner
        folder_obj = drive.CreateFile({'id': gee_folder_id})
        folder_obj.Delete()
        
        print(f"\n✓✓✓ DOWNLOAD ERFOLGREICH! ✓✓✓")
        print(f"Anzahl heruntergeladener Dateien: {downloaded_count}")
        print(f"Zielordner: {DL_folder}")
        print(f"Ordner '{export_folder_name}' wurde aus Google Drive gelöscht.")
        
elif status['state'] == 'FAILED':
    print(f"✗ Export fehlgeschlagen!")
    print(f"Fehler: {status.get('error_message', 'Unbekannter Fehler')}")
else:
    print(f"✗ Unerwarteter Status: {status['state']}")
    print("Bitte Task-Status in Earth Engine prüfen.")

Export gestartet: MHJV2LKEXPRBL5GV4MCFBNWK
Export-Ordner auf GDrive: MBA_GEE_20251118_122349
Überprüfe den Task-Status in der Earth Engine-Taskliste.
Export läuft...
Export läuft...
✓ Export erfolgreich abgeschlossen!

Starte automatischen Download von Google Drive...
✓ Ordner gefunden: MBA_GEE_20251118_122349 (ID: 1DCLdnt4W-0Gf7dTiApARoL6S2eD0Udj6)
Anzahl Kacheln: 1


Download: 100%|██████████| 1/1 [00:02<00:00,  2.41s/it]



✓✓✓ DOWNLOAD ERFOLGREICH! ✓✓✓
Anzahl heruntergeladener Dateien: 1
Zielordner: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs/01_Fire_Metrics/MBA_50000m_3035_DOY_season
Ordner 'MBA_GEE_20251118_122349' wurde aus Google Drive gelöscht.


#### manuell von GDrive in Zielordner herunterladen 

In [ ]:
# # Download processed tiles from GDrive
# drive_folder_url    = "1xuaBU5t9s5bHXCT1pfTGgVST4Fmpt7Zy"

# drive_list = drive.ListFile({'q': f"'{drive_folder_url}' in parents and trashed=false"}).GetList()
# for file in tqdm(drive_list):
#     file_id = drive.CreateFile({'id': file['id']})
#     fname = file["title"]
#     outName = os.path.join(DL_folder, fname)
#     file_id.GetContentFile(outName)
#     file_id.Delete()
#     print(f"{fname} wurde nach {outName} heruntergeladen und aus Google Drive gelöscht.")

  6%|▋         | 1/16 [00:02<00:38,  2.60s/it]

MBA_burned_metrics_yearly_season_500m-0000008448-0000008448.tif wurde nach \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs/01_Fire_Metrics/MBA_500m_3035_DOY_season\MBA_burned_metrics_yearly_season_500m-0000008448-0000008448.tif heruntergeladen und aus Google Drive gelöscht.


 12%|█▎        | 2/16 [00:04<00:32,  2.31s/it]

MBA_burned_metrics_yearly_season_500m-0000008448-0000005632.tif wurde nach \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs/01_Fire_Metrics/MBA_500m_3035_DOY_season\MBA_burned_metrics_yearly_season_500m-0000008448-0000005632.tif heruntergeladen und aus Google Drive gelöscht.


 19%|█▉        | 3/16 [00:06<00:28,  2.19s/it]

...

MBA_burned_metrics_yearly_season_500m-0000000000-0000005632.tif wurde nach \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs/01_Fire_Metrics/MBA_500m_3035_DOY_season\MBA_burned_metrics_yearly_season_500m-0000000000-0000005632.tif heruntergeladen und aus Google Drive gelöscht.


 94%|█████████▍| 15/16 [00:44<00:02,  2.60s/it]

MBA_burned_metrics_yearly_season_500m-0000000000-0000002816.tif wurde nach \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs/01_Fire_Metrics/MBA_500m_3035_DOY_season\MBA_burned_metrics_yearly_season_500m-0000000000-0000002816.tif heruntergeladen und aus Google Drive gelöscht.


100%|██████████| 16/16 [00:46<00:00,  2.92s/it]

MBA_burned_metrics_yearly_season_500m-0000000000-0000000000.tif wurde nach \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs/01_Fire_Metrics/MBA_500m_3035_DOY_season\MBA_burned_metrics_yearly_season_500m-0000000000-0000000000.tif heruntergeladen und aus Google Drive gelöscht.


#### Mosaic aus Kacheln erstellen

In [57]:
import os
import rasterio
from rasterio.merge import merge
from glob import glob

tiles_dir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\01_Fire_Metrics\MBA_500m_3035_DOY_season"
tiles = sorted(glob(os.path.join(tiles_dir, "MBA_burned_metrics_yearly_season_500m-*.tif")))
srcs = [rasterio.open(t) for t in tiles]
mosaic, out_trans = merge(srcs)
out_meta = srcs[0].meta.copy()
out_meta.update({
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_trans,
    "driver": "GTiff",
    # Kompression und Kachelung:
    "compress": "lzw",      # oder "deflate"
    "tiled": True,
    "blockxsize": 512,
    "blockysize": 512,
    # ggf. BIGTIFF nötig bei sehr großen Dateien:
    "BIGTIFF": "YES"
})

out_path = os.path.join(tiles_dir, "MBA_burned_metrics_yearly_season_500m_mosaic.tif")
with rasterio.open(out_path, "w", **out_meta) as dst:
    dst.write(mosaic)

for s in srcs:
    s.close()
print("Mosaic gespeichert unter:", out_path)

Mosaic gespeichert unter: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\01_Fire_Metrics\MBA_500m_3035_DOY_season\MBA_burned_metrics_yearly_season_500m_mosaic.tif


#### rasterbeschreibungen in mosaic einfügen

In [58]:
# Erzeugt Liste aller Bandnamen (2000..2025) und setzt Bandbeschreibungen in der Datei
import os
years = list(range(2000, 2026))
band_names = []
for y in years:
    band_names += [
        f"{y}_burned",
        f"{y}_count",
        f"{y}_doy_1",
        f"{y}_doy_2",
        f"{y}_doy_3",
        f"{y}_doy_4",
        f"{y}_doy_min",
        f"{y}_doy_max",
        f"{y}_doy_span",
        f"{y}_month_first",
        f"{y}_month_last",
        f"{y}_season_length"
    ]

# band_names[0] = "2000_burned", band_names[1] = "2000_count", ..., band_names[155] = "2025_month_last"
# optional: speichere als CSV zur Kontrolle
with open(r"C:\temp\band_list_MBA_burned_metrics_yearly.csv", "w", encoding="utf-8") as f:
    for i, n in enumerate(band_names, start=1):
        f.write(f"{i},{n}\n")

# Optional: Bandbeschreibungen mit rasterio setzen
import rasterio
mosaic_path = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\01_Fire_Metrics\MBA_500m_3035_DOY_season\MBA_burned_metrics_yearly_season_500m_mosaic.tif"
with rasterio.open(mosaic_path, "r+") as dst:
    for i, name in enumerate(band_names, start=1):
        dst.set_band_description(i, name)

# Alternative: mit GDAL (falls rasterio-Version das nicht unterstützt)
# from osgeo import gdal
# ds = gdal.Open(mosaic_path, gdal.GA_Update)
# for i, name in enumerate(band_names, start=1):
#     ds.GetRasterBand(i).SetDescription(name)
# ds = None

In [66]:
# Binäres MBA-Raster erstellen - nur burned-Bänder extrahieren
import os
import rasterio
import numpy as np

# Pfade definieren
mosaic_path = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\01_Fire_Metrics\MBA_500m_3035_DOY_season\MBA_burned_metrics_yearly_season_500m_mosaic.tif"
output_path = os.path.join(os.path.dirname(mosaic_path), "MBA_binary_yearly_500m.tif")

years = list(range(2000, 2026))

# Öffne Mosaik
with rasterio.open(mosaic_path) as src:
    meta = src.meta.copy()
    
    # Extrahiere nur die burned-Bänder (Band 1, 13, 25, ...)
    # Band-Struktur: burned(1), count(2), doy_1(3), ..., season_length(12) -> alle 12 Bänder
    burned_bands = [1 + (i * 12) for i in range(len(years))]
    
    print(f"Extrahiere Bänder: {burned_bands}")
    
    # Lese alle burned-Bänder
    binary_stack = src.read(burned_bands)

# Metadaten aktualisieren
meta.update({
    'count': len(years),
    'dtype': 'uint8',
    'compress': 'lzw',
    'tiled': True,
    'blockxsize': 512,
    'blockysize': 512,
    'BIGTIFF': 'YES'
})

# Schreibe binäres Raster
with rasterio.open(output_path, 'w', **meta) as dst:
    dst.write(binary_stack.astype(np.uint8))
    
    # Setze Bandbeschreibungen
    for i, year in enumerate(years, start=1):
        dst.set_band_description(i, f"{year}_burned")

print(f"\n✓ Binäres MBA-Raster erstellt!")
print(f"Anzahl Bänder: {len(years)}")
print(f"Ausgabedatei: {output_path}")
print(f"Dateigröße: {os.path.getsize(output_path) / (1024**3):.2f} GB")

Extrahiere Bänder: [1, 13, 25, 37, 49, 61, 73, 85, 97, 109, 121, 133, 145, 157, 169, 181, 193, 205, 217, 229, 241, 253, 265, 277, 289, 301]

✓ Binäres MBA-Raster erstellt!
Anzahl Bänder: 26
Ausgabedatei: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\01_Fire_Metrics\MBA_500m_3035_DOY_season\MBA_binary_yearly_500m.tif
Dateigröße: 0.01 GB


#### Code um maximum des count der DOYs zu bekommen

In [ ]:
import os
import csv
import rasterio

mosaic_path = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\01_Fire_Metrics\MBA_500m_3035_DOY\MBA_burned_metrics_yearly_mosaic.tif"
years = list(range(2000, 2026))

# Berechne für jedes Jahr das globale Maximum des "count"-Bands (Band-Index: (Jahr-2000)*6 + 2)
max_per_year = {}
with rasterio.open(mosaic_path) as src:
    for y in years:
        band_index = (y - 2000) * 6 + 2  # 1-based Bandindex in GeoTIFF
        maxv = None
        # block_windows liest Raster blockweise (speicherfreundlich)
        for _, window in src.block_windows(1):
            arr = src.read(band_index, window=window)
            mask = src.read_masks(band_index, window=window)
            if mask is not None:
                vals = arr[mask > 0]
            else:
                vals = arr.ravel()
            if vals.size == 0:
                continue
            local_max = int(vals.max())
            if maxv is None or local_max > maxv:
                maxv = local_max
        max_per_year[y] = 0 if maxv is None else int(maxv)

# Speichere Ergebnis als CSV
out_csv = os.path.join(os.path.dirname(mosaic_path), "max_count_per_year.csv")
with open(out_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["year", "max_count"])
    for y in years:
        writer.writerow([y, max_per_year[y]])

print("Max count pro Jahr berechnet und gespeichert:", out_csv)
print(max_per_year)

Max count pro Jahr berechnet und gespeichert: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\01_Fire_Metrics\MBA_500m_3035_DOY\max_count_per_year.csv
{2000: 2, 2001: 4, 2002: 3, 2003: 3, 2004: 3, 2005: 4, 2006: 3, 2007: 3, 2008: 3, 2009: 3, 2010: 3, 2011: 4, 2012: 3, 2013: 3, 2014: 3, 2015: 3, 2016: 4, 2017: 3, 2018: 4, 2019: 4, 2020: 3, 2021: 4, 2022: 3, 2023: 3, 2024: 3, 2025: 2}
